In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [7]:
import wandb 
wandb.login(key=WB_KEY)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


True

In [8]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F

class ScratchMCQSolver(nn.Module): 
    def __init__(self, input_dim, hidden_dim=128): 
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 5)

    def forward(self, x): 
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits 
vocab_size = 5000 
model_scratch = ScratchMCQSolver(input_dim=vocab_size)
print(model_scratch)

ScratchMCQSolver(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
)


In [9]:
import pandas as pd 
import numpy as np 
import re 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [10]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [11]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [12]:
train_df.fillna("None", inplace=True)

In [13]:
def clean_text(text): 
    text = str(text).lower()
    text = re.sub(r'\s{2,}', '', text)
    return text.strip()

columns_to_clean = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in columns_to_clean: 
    if col in train_df.columns: 
        train_df[col] = train_df[col].apply(clean_text)

print("Data Cleaned")

Data Cleaned


In [14]:
vectorizer = TfidfVectorizer(stop_words='english')
predictions = []
actuals = []

for idx, row in train_df.iterrows():
    corpus = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]

    tfidf_matrix = vectorizer.fit_transform(corpus)
    prompt_vector = tfidf_matrix[0:1]
    options_matrix = tfidf_matrix[1:]

    similarities = cosine_similarity(prompt_vector, options_matrix).flatten()

    labels = ['A', 'B', 'C', 'D', 'E']
    top_3_idx = similarities.argsort()[-3:][::-1]
    top_3_preds = [labels[i] for i in top_3_idx]

    predictions.append(top_3_preds)
    if 'answer' in train_df.columns: 
        actuals.append(row['answer'])


In [15]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual)+1)

    return 0

if actuals: 
    map_3_score = np.mean([apk(a, p) for a, p in zip(actuals, predictions)])
    print(f'Baseline TF-IDF MAP@3: {map_3_score: .4f}')
else: 
    print('No answer column found')

Baseline TF-IDF MAP@3:  0.3260


# Mile 2

In [16]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [17]:
df_pandas=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')
dataset = Dataset.from_pandas(df_pandas)

In [18]:
def combine_text_fn(example): 
    return {"combined_text": f"{example['prompt']} {example['A']}"}

dataset = dataset.map(combine_text_fn)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_prompts(examples): 
    return tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_prompts, batched=True)
input_ids_shape = np.array(tokenized_dataset['input_ids']).shape

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [19]:
model = AutoModel.from_pretrained('bert-base-uncased')
row_0_prompt = dataset[0]['prompt']
inputs_0 = tokenizer(row_0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs_0 = model(**inputs_0)

last_hidden_state = outputs_0.last_hidden_state 

cls_vector = last_hidden_state[0, 0, :5].tolist()

model_att = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text_att = 'Light-ion fusion is a technique.'
inputs_att = tokenizer(text_att, return_tensors='pt')
tokens_att = tokenizer.convert_ids_to_tokens(inputs_att['input_ids'][0])

fusion_idx = tokens_att.index('fusion')

with torch.no_grad(): 
    outputs_att = model_att(**inputs_att)

attention_matrix = outputs_att.attentions[-1][0, 0]
weight_cls_to_fusion = attention_matrix[0, fusion_idx].item()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb_prompt = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
emb_opt_b = st_model.encode(dataset[0]['B'], convert_to_tensor=True)
sim_score = util.cos_sim(emb_prompt, emb_opt_b).item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual) + 1)
    return 0 
labels = ['A', 'B', 'C', 'D', 'E']
tf_idf_vectorizer = TfidfVectorizer(stop_words='english')

minilm_preds = []
tfidf_preds = []
actual_answers = df_pandas['answer'].tolist() if 'answer' in df_pandas.columns else []

all_prompts = df_pandas['prompt'].tolist()
opt_cols = [df_pandas[c].tolist() for c in labels]

prompt_embs = st_model.encode(all_prompts, convert_to_tensor=True)
opt_embs = [st_model.encode(col, convert_to_tensor=True) for col in opt_cols]

for idx, row in df_pandas.iterrows(): 
    corpus = [row['prompt']] + [row[l] for l in labels]
    try: 
        tfidf_mat = tf_idf_vectorizer.fit_transform(corpus)
        sims_tfidf = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).flatten()
        top_3_tfidf = [labels[i] for i in sims_tfidf.argsort()[-3:][::-1]]

    except: 
        top_3_tfidf = ['A', 'B', 'C']
    tfidf_preds.append(top_3_tfidf)

    p_emb = prompt_embs[idx]
    sims_minilm = []
    for o_idx in range(5): 
        sims_minilm.append(util.cos_sim(p_emb, opt_embs[o_idx][idx]).item())

    top_3_minilm = [labels[i] for i in np.argsort(sims_minilm)[-3:][::-1]]
    minilm_preds.append(top_3_minilm)

map3_minilm = np.mean([apk(a, p) for a, p in zip(actual_answers, minilm_preds)])

improvement_count = 0
for a, t_pred, m_pred in zip(actual_answers, tfidf_preds, minilm_preds): 
    if(a in m_pred) and (a not in t_pred): 
        improvement_count += 1

In [22]:
classifier = pipeline('zero-shot-classification', model = 'facebook/bart-large-mnli', device=0 if torch.cuda.is_available() else -1)
prompt_idx_1 = dataset[1]['prompt']
candidates = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

res_softmax = classifier(prompt_idx_1, candidate_labels = candidates, multi_label=False)
top_prob_softmax = res_softmax['scores'][0]

res_sigmoid = classifier(prompt_idx_1, candidate_labels=candidates, multi_label=True)
abs_diff = abs(sum(res_softmax['scores']) - sum(res_sigmoid['scores']))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [23]:
from transformers import AutoModelForSeq2SeqLM

In [24]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
slm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to("cuda:0")
prompt_slm = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."
inputs = tokenizer(prompt_slm, return_tensors="pt").to("cuda:0")
outputs = slm_model.generate(**inputs, max_new_tokens=5)
slm_out = tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Answers Milstone 2

In [25]:
print(f"Q1: Character length at index 51: {len(dataset[51]['combined_text'])}")
print(f"Q2: Vocabulary Size: {tokenizer.vocab_size}")
print(f"Q3: [SEP] Token ID: {tokenizer.convert_tokens_to_ids('[SEP]')}")
print(f"Q4: Geometric shape of input_ids tensor: {input_ids_shape}")
print(f"Q5: Dimensionality of each individual attention head: {768 // 12}")
print(f"Q6: Shape of last_hidden_state tensor: {list(last_hidden_state.shape)}")
print(f"Q7: Sum of first 5 float values in [CLS] vector: {round(sum(cls_vector), 4)}")
print(f"Q8: Attention weight from [CLS] to 'fusion': {round(weight_cls_to_fusion, 4)}")
print(f"Q9: Cosine similarity between prompt and Option B: {round(sim_score, 4)}")
print(f"Q10 Part 1: Final MAP@3 score of MiniLM pipeline: {round(map3_minilm, 4)}")
print(f"Q10 Part 2: Number of questions saved by MiniLM: {improvement_count}")

Q1: Character length at index 51: 614
Q2: Vocabulary Size: 32100
Q3: [SEP] Token ID: 2
Q4: Geometric shape of input_ids tensor: (2000, 128)
Q5: Dimensionality of each individual attention head: 64
Q6: Shape of last_hidden_state tensor: [1, 31, 768]
Q7: Sum of first 5 float values in [CLS] vector: -1.2001
Q8: Attention weight from [CLS] to 'fusion': 0.1025
Q9: Cosine similarity between prompt and Option B: 0.7658
Q10 Part 1: Final MAP@3 score of MiniLM pipeline: 0.4231
Q10 Part 2: Number of questions saved by MiniLM: 488


# Milestone 3

In [30]:
!pip install -q langchain-text-splitters langchain-huggingface faiss-cpu 

In [31]:
!pip install -q langchain-community 

In [32]:
import os 
import re 
import numpy as np
import pandas as pd 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_core.documents import Document 
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import FAISS 

In [33]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')

In [34]:
external_knowledge_corpus = """
Light-ion fusion is a nuclear reaction technique where light atomic nuclei combine to form heavier elements.
Tragedy of Summerhall was a catastrophic fire event in 259 AC that led to the deaths of King Aegon V.
Aegon the Conqueror forged the Iron Throne using the fiery breath of his dragon, Balerion the Black Dread.
Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings.
Transformers rely on multi-head self-attention mechanisms to map contextual text representations dynamically.
"""
def clean_corpus_text(text):
    text = text.strip()
    text = re.sub(r' {2,}', ' ', text)
    text = text.replace("\r", "")
    return text

cleaned_corpus = clean_corpus_text(external_knowledge_corpus)

In [35]:
docs = [Document(page_content=cleaned_corpus, metadata={"source": "external_knowledge_base"})]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(docs)
print(f"Split external knowledge into {len(chunks)} text chunks.")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)

vector_store = FAISS.from_documents(chunks, embeddings_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("FAISS vector database built successfully.")

Split external knowledge into 2 text chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector database built successfully.


In [36]:
augmented_prompts = []

print("Augmenting questions with retrieved context chunks...")
for idx, row in train_df.iterrows():
    question_prompt = row['prompt']
    
    retrieved_docs = retriever.invoke(question_prompt)
    context_str = " ".join([doc.page_content for doc in retrieved_docs])
    
    structured_rag_prompt = f"""Context: {context_str}
Question: {question_prompt}
Options:
A) {row['A']}
B) {row['B']}
C) {row['C']}
D) {row['D']}
E) {row['E']}
Analyze the context above and pick the top 3 most likely correct answers in ranked order."""

    augmented_prompts.append(structured_rag_prompt)

train_df['augmented_rag_prompt'] = augmented_prompts

print("\n=== Sample Augmented RAG Prompt (Row 0) ===")
print(train_df['augmented_rag_prompt'].iloc[0])

train_df.to_csv("train_augmented_rag.csv", index=False)
print("\nSaved RAG-augmented dataset to train_augmented_rag.csv")

Augmenting questions with retrieved context chunks...

=== Sample Augmented RAG Prompt (Row 0) ===
Context: Light-ion fusion is a nuclear reaction technique where light atomic nuclei combine to form heavier elements.
Tragedy of Summerhall was a catastrophic fire event in 259 AC that led to the deaths of King Aegon V.
Aegon the Conqueror forged the Iron Throne using the fiery breath of his dragon, Balerion the Black Dread.
Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings. Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings.
Transformers rely on multi-head self-attention mechanisms to map contextual text representations dynamically.
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Options:
A) Martin Heidegger believes that humans exist within a time continuum that 

In [37]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [38]:
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wandb
import numpy as np 

label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {v: k for k, v in label2id.items()}

X_text = (train_df['prompt'] + " " + train_df['A'] + " " + train_df['B'] + " " + train_df['C'] + " " + train_df['D'] + " " + train_df['E']).tolist()
X_tfidf = vectorizer.fit_transform(X_text).toarray()
y_scratch = np.array([label2id[ans] for ans in train_df['answer']])

class MCQScratchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

scratch_dataset = MCQScratchDataset(X_tfidf, y_scratch)
scratch_loader = DataLoader(scratch_dataset, batch_size=16, shuffle=True)

wandb.init(project="smart-mcq-solver", name="Model-1-Scratch-MLP")

input_dim = X_tfidf.shape[1]
model_scratch = ScratchMCQSolver(input_dim=input_dim, hidden_dim=128).to("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_scratch.train()

for epoch in range(10):
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_x, batch_y in scratch_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_scratch(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        
    acc = correct / total
    avg_loss = total_loss / len(scratch_loader)
    wandb.log({"epoch": epoch + 1, "loss": avg_loss, "accuracy": acc})
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f} - Accuracy: {acc:.4f}")

wandb.finish()

Epoch 1/10 - Loss: 1.2103 - Accuracy: 0.7645
Epoch 2/10 - Loss: 0.1644 - Accuracy: 1.0000
Epoch 3/10 - Loss: 0.0278 - Accuracy: 1.0000
Epoch 4/10 - Loss: 0.0116 - Accuracy: 1.0000
Epoch 5/10 - Loss: 0.0063 - Accuracy: 1.0000
Epoch 6/10 - Loss: 0.0040 - Accuracy: 1.0000
Epoch 7/10 - Loss: 0.0027 - Accuracy: 1.0000
Epoch 8/10 - Loss: 0.0020 - Accuracy: 1.0000
Epoch 9/10 - Loss: 0.0015 - Accuracy: 1.0000
Epoch 10/10 - Loss: 0.0012 - Accuracy: 1.0000


accuracy,▁█████████
epoch,▁▂▃▃▄▅▆▆▇█
loss,█▂▁▁▁▁▁▁▁▁
accuracy,1
epoch,10
loss,0.0012


In [39]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from dataclasses import dataclass
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from typing import Optional, Union
import torch

In [42]:
from datasets import Dataset as HFDataset

hf_train_ds = HFDataset.from_pandas(train_df)

In [46]:
MODEL_2_NAME = "microsoft/deberta-v3-base"
tokenizer_m2 = AutoTokenizer.from_pretrained(MODEL_2_NAME)

def preprocess_mcq(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"Option A: {a}", f"Option B: {b}", f"Option C: {c}", f"Option D: {d}", f"Option E: {e}"]
        for a, b, c, d, e in zip(examples['A'], examples['B'], examples['C'], examples['D'], examples['E'])
    ]
    
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized = tokenizer_m2(first_sentences, second_sentences, truncation=True, max_length=256)

    return {
        k: [v[i:i+5] for i in range(0, len(v), 5)]
        for k, v in tokenized.items()
    }

hf_train_ds = HFDataset.from_pandas(train_df)
encoded_ds = hf_train_ds.map(preprocess_mcq, batched=True, remove_columns=hf_train_ds.column_names)
encoded_ds = encoded_ds.add_column("label", [label2id[ans] for ans in train_df['answer']])

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features]
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flat_features = [
            [{k: v[i] for k, v in feature.items()} for i in range(num_choices)]
            for feature in features
        ]
        flat_features = sum(flat_features, [])
        
        batch = self.tokenizer.pad(
            flat_features,
            padding=self.padding,
            max_length=self.max_length,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

model_2 = AutoModelForMultipleChoice.from_pretrained(MODEL_2_NAME)

training_args_m2 = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="no",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
    max_grad_norm=1.0,
    warmup_ratio=0.1,
    report_to="wandb",
    run_name="Model-2-DeBERTa-v3-Full"
)

trainer_m2 = Trainer(
    model=model_2,
    args=training_args_m2,
    train_dataset=encoded_ds,
    processing_class=tokenizer_m2,
    data_collator=DataCollatorForMultipleChoice(tokenizer=tokenizer_m2),
)

trainer_m2.train()
wandb.finish()

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,12.855273
20,12.907422
30,12.908594
40,12.874023
50,12.894336
60,12.902148
70,12.834375
80,12.882617
90,12.855273
100,12.853125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▃▄▆███▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,▂▄▄▃▃▄▂▃▂▂▂█▄▄▄▅▇▁▃▁▄▆▄▂▆▅▇▃▃▄▃▃▂▁▃▃▄
total_flos,1347932063118360.0
train/epoch,3
train/global_step,375
train/grad_norm,10.03125
train/learning_rate,0.0
train/loss,12.9332


# RESTART

In [52]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
import wandb
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset as HFDataset

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').fillna('None')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [53]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_text = (train_df['prompt'] + " " + train_df['A'] + " " + train_df['B'] + " " + train_df['C'] + " " + train_df['D'] + " " + train_df['E']).tolist()
X_test_text = (test_df['prompt'] + " " + test_df['A'] + " " + test_df['B'] + " " + test_df['C'] + " " + test_df['D'] + " " + test_df['E']).tolist()

X_train_tfidf = vectorizer.fit_transform(X_train_text).toarray()
X_test_tfidf = vectorizer.transform(X_test_text).toarray()
y_train = np.array([label2id[ans] for ans in train_df['answer']])

class MCQScratchDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

scratch_loader = DataLoader(MCQScratchDataset(X_train_tfidf, y_train), batch_size=16, shuffle=True)

class ScratchMCQSolver(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 5)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

device = "cuda" if torch.cuda.is_available() else "cpu"
model_scratch = ScratchMCQSolver(input_dim=X_train_tfidf.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

wandb.init(project="smart-mcq-solver", name="Model-1-Scratch-MLP")
model_scratch.train()

for epoch in range(5): 
    total_loss, correct, total = 0.0, 0, 0
    for batch_x, batch_y in scratch_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_scratch(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (torch.argmax(outputs, dim=1) == batch_y).sum().item()
        total += batch_y.size(0)
    
    wandb.log({"epoch": epoch + 1, "loss": total_loss / len(scratch_loader), "accuracy": correct / total})
wandb.finish()

accuracy,▁████
epoch,▁▃▅▆█
loss,█▂▁▁▁
accuracy,1
epoch,5
loss,0.0074


In [54]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_mcq(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"A: {a}", f"B: {b}", f"C: {c}", f"D: {d}", f"E: {e}"]
        for a, b, c, d, e in zip(examples['A'], examples['B'], examples['C'], examples['D'], examples['E'])
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, max_length=128) 
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

hf_train_ds = HFDataset.from_pandas(train_df)
encoded_train_ds = hf_train_ds.map(preprocess_mcq, batched=True, remove_columns=hf_train_ds.column_names)
encoded_train_ds = encoded_train_ds.add_column("label", y_train.tolist())

hf_test_ds = HFDataset.from_pandas(test_df)
encoded_test_ds = hf_test_ds.map(preprocess_mcq, batched=True, remove_columns=hf_test_ds.column_names)

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flat_features = [[{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features]
        flat_features = sum(flat_features, [])
        
        batch = self.tokenizer.pad(flat_features, padding=self.padding, max_length=self.max_length, pad_to_multiple_of=self.pad_to_multiple_of, return_tensors="pt")
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [57]:
model_2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args_m2 = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="no",
    learning_rate=1e-5,
    warmup_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, 
    num_train_epochs=3,
    weight_decay=0.01,
    adam_epsilon=1e-6,
    logging_steps=10,
    max_grad_norm=0.5,           
    fp16=False,                  
    report_to="wandb",
    run_name="Model-2-DeBERTa-v3-Full"
)

trainer_m2 = Trainer(
    model=model_2,
    args=training_args_m2,
    train_dataset=encoded_train_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer_m2.train()
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
10,25.734766
20,25.730078
30,25.662891
40,25.610352
50,25.641406
60,26.301562
70,24.612305
80,25.760547
90,26.029688
100,25.826367


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

train/epoch,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
train/global_step,▁▁▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇██
train/grad_norm,▁▁▁▄█▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▂▃▅▆██▇▇▆▅▅▄▄▃▃▂▂▁
train/loss,▆▆▅▅▅█▁▆▇▆▆▆▁▆▆▆▆▆
total_flos,1321363775268960.0
train/epoch,3
train/global_step,189
train/grad_norm,10.26562
train/learning_rate,0.0
train/loss,25.79316


In [ ]:
preds_m2 = trainer_m2.predict(encoded_test_ds)
logits_m2 = preds_m2.predictions

final_predictions_m2 = []
labels = ['A', 'B', 'C', 'D', 'E']

for row_logits in logits_m2:
    top_3_indices = np.argsort(row_logits)[-3:][::-1]
    final_predictions_m2.append(" ".join([labels[i] for i in top_3_indices]))

# Detect ID column name
id_col = 'ID' if 'ID' in test_df.columns else 'id' if 'id' in test_df.columns else test_df.index

# Save Submission
submission_df_m2 = pd.DataFrame({
    'ID': test_df[id_col] if id_col in test_df.columns else id_col,
    'Prediction': final_predictions_m2
})

submission_df_m2.to_csv('submission_model2.csv', index=False)
print("Done!")
